# Colabで無料GPUを使う - 画像生成スターター（かーくん用）

このファイルが「手順書ファイル」です。**上から順に ▶ を押すだけ**で進みます。

## 始める前に（1回だけ）
1. このファイルを Google Colab で開く
2. 上のメニュー **ランタイム → ランタイムのタイプを変更**
3. ハードウェア アクセラレータで **T4 GPU** を選んで「保存」
4. 下の「① GPU確認」の ▶ を押して、**Tesla T4** と出れば準備完了

> iPhoneのブラウザでも開けますが、操作はPCのブラウザが圧倒的に楽です。
> クレカ登録は不要です。普段使いのGoogleアカウントでOKです。


## ① GPU確認

In [ ]:
# GPUが本当に使えるか確認します
!nvidia-smi

import torch
if torch.cuda.is_available():
    print("GPU名:", torch.cuda.get_device_name(0))
    print("OK：GPUが使えます。このまま下へ進んでください。")
else:
    print("GPUが有効になっていません。")
    print("ランタイム → ランタイムのタイプを変更 → T4 GPU を選んで保存し、もう一度このセルを実行してください。")


## ② 必要なものを入れる（初回だけ・約1〜2分）
毎回 Colab を開き直したら、ここももう一度実行してください（Colabは終了するとリセットされます）。

In [ ]:
# 画像生成に必要なライブラリを入れます
!pip -q install diffusers transformers accelerate safetensors
print("インストール完了。次へ進んでください。")


## ③ 生成したい絵を決める（ここだけ書き換える）

- `PROMPT` ：英語の方がうまく伝わります。日本語で思いついたら、Museに「画像生成用の英語プロンプトにして」と頼んでください。
- 縦長にしたいときは `WIDTH, HEIGHT = 576, 832` のように変えます（9:16風）。
- `SEED` を変えると、同じ指示でも違う絵になります。


In [ ]:
# ===== ここを書き換える（v2設定：空白ページ強調・落書き抑制） =====
PROMPT = "top view of a wooden desk with open notebooks with completely blank empty white pages, a cup of black coffee, a smartphone with a blank black screen, soft morning sunlight, clean flat illustration, calm and cozy, no people"
NEGATIVE_PROMPT = "blurry, low quality, distorted, text, letters, handwriting, scribbles, drawings on paper, abstract shapes, watermark, logo, face, deformed hands, extra fingers"
WIDTH, HEIGHT = 832, 576   # 横長 16:9っぽい比率。縦長は 576, 832
SEED = 7
STEPS = 30                 # 大きいほど丁寧だが遅くなる（20〜30でOK）
# =============================
print("設定OK（v2）:", PROMPT[:60], "...")


## ④ モデルを読み込んで生成する（初回はモデルのダウンロードで数分、2枚目からは1枚 約20〜40秒が目安）

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()

generator = torch.Generator("cuda").manual_seed(SEED)
image = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_inference_steps=STEPS,
    generator=generator,
).images[0]

out = "/content/kakun_output.png"
image.save(out)
display(image)
print("保存しました:", out)
print("画像の上で右クリック（スマホは長押し）で保存できます。次のセルでまとめてダウンロードもできます。")


## ⑤ もう1枚作る
`SEED` の数字だけ変えて、このセルを実行すると別バージョンが出ます。モデル再読み込みなしなので速いです。

In [ ]:
import torch
from IPython.display import display

SEED2 = SEED + 1  # 変えたいときは数字を直接書き換えてOK
generator = torch.Generator("cuda").manual_seed(SEED2)
image2 = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_inference_steps=STEPS,
    generator=generator,
).images[0]
out2 = f"/content/kakun_output_{SEED2}.png"
image2.save(out2)
display(image2)
print("保存しました:", out2)


## ⑥ パソコンにダウンロードする / Googleドライブに保存する（お好みで）

In [ ]:
# パソコンへダウンロード（ブラウザが許可を求めたら許可してください）
from google.colab import files
files.download("/content/kakun_output.png")
print("ダウンロードを開始しました。")


In [ ]:
# Googleドライブに残したいときだけ実行（任意）
# 実行するとドライブへのアクセス許可が表示されます。
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil, glob
# for f in glob.glob("/content/kakun_output*.png"):
#     shutil.copy(f, "/content/drive/MyDrive/" + f.split("/")[-1])
# print("ドライブへコピーしました")
print("ドライブ保存は任意です。使うときは上の行頭の # を外してください。")


## 困ったとき・知っておくこと
- **Tesla T4 と出ない／GPUが割り当てられない**：無料枠の混雑時によくあります。時間を置いて開き直してください。有料化は不要です。
- **途中で切れた**：無料Colabは放置すると切れます。②から順に実行し直せば復帰します。
- **文字やロゴが変に入る**：この方式は文字生成が苦手です。文字は後からCanva等で載せる前提が安全です（今までの背景素材セットと同じ方針）。
- **無料枠の上限**：決まった時間数は公表されていません。使いすぎると一時的にGPUが使えなくなりますが、普段使いのGoogleアカウント自体が止まるものではありません。心配なら作業用に分けたアカウントでもOKです。
- **動画について**：このノートは画像生成用です。T4で動画生成モデルも動かせますが、数秒の短い動画でも数分かかり画質も今のMuse生成とは別物です。動画の本編は今まで通り、画像はColabで枚数を稼ぐ、という使い分けが現実的です。
